# Tâche 1 : Question-réponse avec un modèle génératif pré‑entraîné sans adaptation

L'objectif de cette tâche est d'observer les limites d’un modèle de langage **pré‑entraîné** lorsqu’on lui pose des questions sur des sujets **potentiellement absents** de ses données d’origine.

Dans ce *notebook*, vous faites **uniquement de l’inférence** (c.-à-d. générer des textes) avec un modèle déjà pré-entraîné. Les paires **questions-réponses** pour évaluer le modèle vous sont fournies (voir fichier *data/test_questions.json*). Cette tâche consiste à utiliser les questions comme *prompt*, générer des réponses avec le modèle non modifié, puis évaluer la pertinence des résultats à partir des réponses de référence. Aucun entraînement de modèle n'est effectué pour cette tâche.

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

In [1]:
import json
import os
import time
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers.pipelines.pt_utils import KeyDataset
import torch

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement du modèle et du tokeniser

In [2]:
model_name = "EleutherAI/pythia-410m"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="saved_models/pythia-410m")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Configuration essentielle pour la génération en batch
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.bfloat16, cache_dir="saved_models/pythia-410m")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 292/292 [00:00<00:00, 943.75it/s] 


## 2. Fonctions de test question-réponse

In [3]:
def build_prompt(question, context=None):
    few_shot = (
        "Question: What mechanism allows a Transformer model to weigh the importance of different words in a sequence?\n"
        "Answer: The self-attention mechanism allows a Transformer to weigh the importance of different tokens relative to each other.\n\n"
        "Question: What is the purpose of positional encodings in the Transformer architecture?\n"
        "Answer: Positional encodings inject information about the relative or absolute position of tokens in the sequence.\n\n"
    )
    if context:
        return f"Answer concisely and factually.\n\n{few_shot}Context: {context}\nQuestion: {question}\nAnswer:"
    return f"Answer concisely and factually.\n\n{few_shot}Question: {question}\nAnswer:"

def clean_generation(generated_text):
    text = generated_text.strip()
    for stop_token in ["\nQuestion:", "\nQ:", "\n###", "\n\n"]:
        if stop_token in text:
            text = text.split(stop_token)[0].strip()
    return text

## 3. Génération des réponses aux questions avec le modèle

In [4]:
test_fn = "data/test_questions.json"

raw_dataset = load_dataset("json", data_files=test_fn)["train"]

dataset = raw_dataset.map(
    lambda example: {"prompt": build_prompt(example["question"])},
)
print(f"Nombre de questions chargées dans le Dataset : {len(dataset)}")

Nombre de questions chargées dans le Dataset : 50


In [11]:
pipe_dataset = KeyDataset(dataset, "prompt")

raw_outputs = []
start_time = time.time()


generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

for out in generator(
    pipe_dataset,
    batch_size=16,
    max_new_tokens=264,
    do_sample=True,
    repetition_penalty=1.05,
    top_p=0.7,
    temperature=0.5,
    stop_strings=["\n", "Question:"],
    tokenizer=tokenizer,
    return_full_text=False
):
    raw_outputs.append(out[0]["generated_text"])

print(f"Génération en batch complétée en {time.time() - start_time:.2f} secondes !")

results = []
for sample, raw_gen in zip(dataset, raw_outputs):
    results.append({
        "Q": sample["question"],
        "A": clean_generation(raw_gen),
        "Expected": sample.get("answer", "")
    })

out_dir = "result"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "t1_results.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Fichier de résultats sauvegardé ({len(results)} entrées) : {out_path}")

Génération en batch complétée en 3.27 secondes !
Fichier de résultats sauvegardé (50 entrées) : result\t1_results.json


## 4. Évaluation des résultats

In [12]:
import re
from collections import Counter
import evaluate
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def tokenize_no_stopwords(text: str):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]


def compute_word_metrics(results: list[dict]):
    precisions, recalls, f1s = [], [], []

    for item in results:
        pred_tokens = tokenize_no_stopwords(item["A"])
        ref_tokens = tokenize_no_stopwords(item["Expected"])

        common = Counter(pred_tokens) & Counter(ref_tokens)
        num_same = sum(common.values())

        if len(pred_tokens) == 0 or len(ref_tokens) == 0 or num_same == 0:
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            continue

        p = num_same / len(pred_tokens)
        r = num_same / len(ref_tokens)
        f1 = (2 * p * r) / (p + r)

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    n = len(results)
    avg_p = (sum(precisions) / n) * 100 if n > 0 else 0.0
    avg_r = (sum(recalls) / n) * 100 if n > 0 else 0.0
    avg_f1 = (sum(f1s) / n) * 100 if n > 0 else 0.0
    return avg_p, avg_r, avg_f1


squad = evaluate.load("squad")

squad_predictions = [{"id": str(i), "prediction_text": item["A"]} for i, item in enumerate(results)]
squad_references = [{"id": str(i), "answers": {"text": [item["Expected"]], "answer_start": [0]}} for i, item in enumerate(results)]

metrics = squad.compute(predictions=squad_predictions, references=squad_references)
word_precision, word_recall, word_f1 = compute_word_metrics(results)

print("=" * 60)
print(" RÉSULTATS EVALUATE:")
print(f" Exact Match (SQuAD) : {metrics['exact_match']:.2f}%")
print(f" F1-Score (SQuAD)    : {metrics['f1']:.2f}%")
print("-" * 60)
print(f" Précision (Mots)    : {word_precision:.2f}%")
print(f" Rappel (Mots)       : {word_recall:.2f}%")
print(f" F1-Score (Mots)     : {word_f1:.2f}%")
print("=" * 60)

 RÉSULTATS EVALUATE:
 Exact Match (SQuAD) : 0.00%
 F1-Score (SQuAD)    : 23.22%
------------------------------------------------------------
 Précision (Mots)    : 29.37%
 Rappel (Mots)       : 19.35%
 F1-Score (Mots)     : 20.89%


In [13]:
print("Exemples de réponses générées :")
for sample in results[:3]:
    print("-" * 50)
    print(f"Q        : {sample['Q']}")
    print(f"Expected : {sample['Expected']}")
    print(f"A (gen)  : {sample['A']}")

Exemples de réponses générées :
--------------------------------------------------
Q        : What is the context window size of GPT-3?
Expected : GPT-3 has a context window size of 2,048 tokens.
A (gen)  : The context window size determines the number of tokens that can be processed at the same time.
--------------------------------------------------
Q        : Explain what is a transformer architecture.
Expected : The transformer architecture is a neural network based on a deep learning model that was introduced in 2017, used in natural language processing (NLP) systems.
A (gen)  : A transformer architecture is an architecture that can be used to transform data from one format into another.
--------------------------------------------------
Q        : Define what is a generative pre-trained transformer (GPT).
Expected : A generative pre-trained transformer is a type of generative large language model that is pre-trained with an enormous and diverse text corpus in datasets, followed b